# Mission 2: Market-Maker in the Arena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_02_marketmaker/notebook.ipynb)

**Learning objectives**
- Understand the economics of market-making: earning the spread vs. adverse selection
- Build a live agent on the official `RemoteAgent` SDK
- Measure inventory risk and attribute PnL (spread income vs. inventory mark-to-market)
- Iterate on quoting logic and climb the live leaderboard

## Background

A **market-maker** continuously posts bid and ask limit orders, earning the spread when both sides
fill. The risk: an informed trader knows where prices are going and hits your stale quotes — **adverse
selection**. Good market-makers quote a spread wide enough to cover it, **manage inventory**, and
react quickly. The Arena is a discrete-time limit order book; each tick you get the book state and
return orders.

> ## ▶ Connecting to the **live** Arena
>
> The supported way to run a live agent is the **`RemoteAgent` SDK** — it speaks the deployed
> server's protocol for you:
>
> ```python
> # pip install convexpi-arena websockets
> from convexpi.arena.client import RemoteAgent, MarketState
>
> class MyMM(RemoteAgent):
>     def on_tick(self, state: MarketState) -> list[dict]:
>         if state.mid is None:
>             return []
>         orders = [self.cancel(o["order_id"]) for o in state.my_open_orders]
>         half = state.mid * 8 / 1e4
>         orders += [self.limit("buy",  round(state.mid - half), 5),
>                    self.limit("sell", round(state.mid + half), 5)]
>         return orders
>
> MyMM("your-handle", server="wss://arena-production-e3f1.up.railway.app").start()
> ```
>
> Watch your PnL + **maker %** at **[/compete/arena-book](https://convexpi.ai/compete/arena-book)** (a
> *real recorded order book*). More: **[/lessons/market-making](https://convexpi.ai/lessons/market-making)**
> and **[/exchange](https://convexpi.ai/exchange)**. The cells below build the same intuition step by
> step with a telemetry-collecting harness.

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q convexpi-arena websockets

In [ ]:
import asyncio, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import websockets
from convexpi.arena.client import RemoteAgent, MarketState
plt.style.use('seaborn-v0_8-darkgrid')
print('ready')

In [ ]:
# Arena connection config
ARENA_WS_URL = 'wss://arena-production-e3f1.up.railway.app'   # or 'ws://localhost:8765' for a local server
AGENT_ID     = 'mm_student_01'                                # your unique name on the leaderboard

## Part 1: The agent contract

Subclass `RemoteAgent` and implement `on_tick(state)`, returning a list of orders. Each tick the
server hands you a read-only `MarketState`:

| field | meaning |
|---|---|
| `state.best_bid` / `state.best_ask` | top of book, in **integer cents** (`None` if empty) |
| `state.mid` / `state.spread` | convenience properties |
| `state.last_price` | last trade price (cents) |
| `state.position`, `state.cash` | your signed inventory and cash (cents) |
| `state.my_open_orders` | `[{"order_id","side","price","qty"}, …]` |

Order helpers: `self.limit(side, price, qty)`, `self.market_order(side, qty)`, `self.cancel(order_id)`.

In [ ]:
class NaiveMarketMaker(RemoteAgent):
    """Quote a fixed half-spread around the mid, re-quoting each tick. No inventory management."""
    def __init__(self, agent_id="mm_naive", half_spread=5, qty=10, **kw):
        super().__init__(agent_id, **kw)
        self.half_spread = half_spread
        self.qty = qty

    def on_tick(self, state: MarketState):
        if state.best_bid is None or state.best_ask is None:
            return []
        mid = (state.best_bid + state.best_ask) // 2
        orders = [self.cancel(o["order_id"]) for o in state.my_open_orders]
        orders.append(self.limit("buy",  mid - self.half_spread, self.qty))
        orders.append(self.limit("sell", mid + self.half_spread, self.qty))
        return orders

print("NaiveMarketMaker defined.")

## Part 2: Run it

A small harness connects a `RemoteAgent` to the Arena, runs it for `n_ticks`, and records telemetry
for analysis. (For real live play, just call `agent.start()`.)

In [ ]:
async def run_agent(agent, ws_url, agent_id, n_ticks=200):
    """Run a RemoteAgent for n_ticks and return a telemetry DataFrame."""
    telem, init = [], 0.0
    try:
        async with websockets.connect(ws_url) as ws:
            await ws.send(json.dumps({"type": "join", "agent_id": agent_id}))
            welcome = json.loads(await ws.recv())
            init = welcome.get("initial_cash_cents", 100_000) / 100
            n = 0
            async for raw in ws:
                msg = json.loads(raw)
                if msg.get("type") != "tick":
                    continue
                state = MarketState(
                    tick=msg["tick"], best_bid=msg["best_bid"], best_ask=msg["best_ask"],
                    last_price=msg["last_price"], depth=msg["depth"],
                    recent_trades=msg["recent_trades"], position=msg["position"],
                    cash=msg["cash"], my_open_orders=msg["my_open_orders"])
                orders = agent.on_tick(state) or []
                await ws.send(json.dumps({"type": "orders", "tick": state.tick, "orders": orders}))
                telem.append({
                    "tick": state.tick,
                    "pnl": (state.pnl_dollars or init) - init,
                    "position": state.position,
                    "last_price": (state.last_price or 0) / 100,
                    "spread": (state.spread or 0) / 100,
                    "trades": len(state.recent_trades),
                })
                n += 1
                if n % 50 == 0:
                    print(f"  tick {state.tick:4d}  PnL=${telem[-1]['pnl']:+7.2f}  pos={state.position:+4d}")
                if n >= n_ticks:
                    break
    except Exception as e:
        print(f"Connection error: {e}\nIs the Arena running at {ws_url}?")
    return pd.DataFrame(telem)


agent = NaiveMarketMaker(half_spread=5, qty=10)
print(f"Connecting to {ARENA_WS_URL} as '{AGENT_ID}'...")
df = await run_agent(agent, ARENA_WS_URL, AGENT_ID, n_ticks=200)
print(f"\nDone. Collected {len(df)} ticks.")

## Part 3: Analyse telemetry

In [ ]:
if df.empty:
    print("No data — check the Arena connection above.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
    axes[0].plot(df['tick'], df['pnl'], lw=1.5, color='steelblue')
    axes[0].axhline(0, color='grey', lw=0.5); axes[0].set_title('PnL (dollars)')
    axes[1].plot(df['tick'], df['position'], lw=1.2, color='darkorange')
    axes[1].axhline(0, color='grey', lw=0.5); axes[1].set_title('Inventory (shares)')
    axes[2].plot(df['tick'], df['last_price'], lw=1, color='grey')
    axes[2].set_title('Market price ($)'); axes[2].set_xlabel('Tick')
    plt.tight_layout(); plt.show()
    print(f"\nFinal PnL: ${df['pnl'].iloc[-1]:.2f}   Max |inventory|: {df['position'].abs().max()} shares")

### Diagnosing performance

| Symptom | Likely cause | Fix |
|---|---|---|
| PnL drifts negative despite trades | Adverse selection | Widen the spread or skew against inventory |
| Inventory grows one-sided | No inventory management | Add a position limit and skew quotes |
| Very few fills | Spread too wide | Tighten the spread or quote larger |

## Part 4: Inventory-aware market maker

Skew quotes by inventory: if you're long, lower both quotes so you sell more eagerly; cap the
absolute position; and size down as inventory grows.

In [ ]:
class InventoryAwareMarketMaker(RemoteAgent):
    def __init__(self, agent_id="mm_inv", half_spread=6, qty=10,
                 skew_per_share=0.5, max_position=50, **kw):
        super().__init__(agent_id, **kw)
        self.half_spread = half_spread
        self.qty = qty
        self.skew_per_share = skew_per_share
        self.max_position = max_position

    def on_tick(self, state: MarketState):
        if state.best_bid is None or state.best_ask is None:
            return []
        pos = state.position
        orders = [self.cancel(o["order_id"]) for o in state.my_open_orders]
        if abs(pos) >= self.max_position:
            return orders
        mid = (state.best_bid + state.best_ask) / 2
        skewed = mid - pos * self.skew_per_share
        bid, ask = round(skewed - self.half_spread), round(skewed + self.half_spread)
        if bid >= state.best_ask or ask <= state.best_bid:
            return orders
        q = max(1, round(self.qty * max(0.2, 1 - abs(pos) / self.max_position)))
        orders.append(self.limit("buy", bid, q))
        orders.append(self.limit("sell", ask, q))
        return orders

print("InventoryAwareMarketMaker defined.")

In [ ]:
agent2 = InventoryAwareMarketMaker(half_spread=6, qty=10, skew_per_share=0.5, max_position=50)
print(f"Connecting as '{AGENT_ID}_v2'...")
df2 = await run_agent(agent2, ARENA_WS_URL, AGENT_ID + '_v2', n_ticks=200)
if not df2.empty:
    print(f"\nNaive final PnL:    ${df['pnl'].iloc[-1]:.2f}")
    print(f"Improved final PnL: ${df2['pnl'].iloc[-1]:.2f}")

## Part 5: PnL attribution

Split PnL into inventory mark-to-market vs. the residual (spread income).

In [ ]:
def attribute_pnl(df):
    if df.empty:
        return df
    df = df.copy()
    price_change = df['last_price'].diff().fillna(0)
    df['mtm_pnl'] = df['position'].shift(1).fillna(0) * price_change
    df['spread_pnl'] = df['pnl'] - df['mtm_pnl'].cumsum()
    return df

if not df2.empty:
    attr = attribute_pnl(df2)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(attr['tick'], attr['pnl'], label='Total PnL', lw=2)
    ax.plot(attr['tick'], attr['spread_pnl'], label='Spread income', lw=1.5, linestyle='--')
    ax.plot(attr['tick'], attr['mtm_pnl'].cumsum(), label='Inventory MTM', lw=1.5, linestyle=':')
    ax.axhline(0, color='grey', lw=0.5); ax.set_title('PnL attribution ($)'); ax.set_xlabel('Tick'); ax.legend()
    plt.tight_layout(); plt.show()

## Part 6: Challenges

- **A (easy):** volatility-adaptive spread — widen when recent prices are choppy (starter below).
- **B (medium):** back off after a streak of adverse fills.
- **C (hard):** estimate informed-flow pressure from buy/sell trade imbalance and widen when it's high.

In [ ]:
class VolAdaptiveMM(InventoryAwareMarketMaker):
    def __init__(self, k=0.5, **kw):
        super().__init__(**kw)
        self.k = k
        self._prices = []

    def on_tick(self, state: MarketState):
        if state.last_price:
            self._prices.append(state.last_price)
            if len(self._prices) > 20:
                self._prices.pop(0)
            if len(self._prices) >= 5:
                self.half_spread = max(3, round(self.k * float(np.std(self._prices)) / 100))
        return super().on_tick(state)

print("VolAdaptiveMM defined. Run it with run_agent(), same as the others.")

## Wrap-up

1. **Spread is not free money** — it compensates for adverse selection.
2. **Inventory is risk** — a maker who ignores it is just a directional trader with extra steps.
3. **Skewing quotes** is the textbook fix; **volatility** governs the optimal spread.

→ Next: Mission 3, Alpha Discovery.